In [ ]:
import torch
import torch.nn as nn

In [ ]:
vocab_size = 10
n_embed = 3

token_embedding_table = nn.Embedding(vocab_size, n_embed)
token_embedding_table

Embedding(10, 3)

In [ ]:
token_embedding_table.weight

Parameter containing:
tensor([[-0.4056,  0.2983, -1.0332],
        [-0.9273,  1.2489, -0.5911],
        [-0.2513, -1.3201,  0.9937],
        [-1.1806,  0.4658,  0.7503],
        [ 0.6185,  2.0790, -0.9532],
        [-1.6679,  0.7966,  0.1917],
        [ 1.5560,  0.4988,  0.4995],
        [ 0.2361,  0.1229, -0.4218],
        [ 1.0748,  0.3739,  1.9104],
        [-0.1961, -0.1405,  0.3871]], device='cuda:0', requires_grad=True)

# **Position Embedding Table**

In [ ]:
B, T, C = 2, 5, 3
vocab_size = 10
block_size = 8

token_embedding_table = nn.Embedding(vocab_size, C)

position_embedding_table = nn.Embedding(block_size, C)

idx = torch.randint(0, vocab_size, (B, T))

tok_emb = token_embedding_table(idx)
tok_emb

tensor([[[ 0.5608,  0.5050, -0.4166],
         [ 0.7977, -0.0552,  0.8632],
         [-0.9478, -0.6658,  1.3686],
         [-0.0802,  0.4182, -0.1211],
         [ 0.5608,  0.5050, -0.4166]],

        [[ 0.6868,  0.4488, -0.9064],
         [-0.0802,  0.4182, -0.1211],
         [ 0.7977, -0.0552,  0.8632],
         [-0.9886,  0.2502, -1.0418],
         [ 0.6868,  0.4488, -0.9064]]], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)

In [ ]:
tok_emb.shape

torch.Size([2, 5, 3])

In [ ]:
pos = torch.arange(0, T, dtype=torch.long)
print(pos)
pos_emb = position_embedding_table(pos)
pos_emb

tensor([0, 1, 2, 3, 4], device='cuda:0')


tensor([[ 1.7879,  1.3089,  0.8055],
        [ 0.7627,  1.0562,  1.0069],
        [ 0.6070, -0.1127,  1.8435],
        [ 1.0788, -0.3481,  0.6650],
        [-1.4121,  1.4785, -1.4763]], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)

In [ ]:
X = tok_emb + pos_emb
X

tensor([[[ 2.3487,  1.8139,  0.3889],
         [ 1.5604,  1.0010,  1.8701],
         [-0.3408, -0.7785,  3.2121],
         [ 0.9985,  0.0701,  0.5439],
         [-0.8513,  1.9835, -1.8928]],

        [[ 2.4748,  1.7577, -0.1009],
         [ 0.6825,  1.4744,  0.8859],
         [ 1.4047, -0.1678,  2.7066],
         [ 0.0902, -0.0979, -0.3768],
         [-0.7252,  1.9273, -2.3827]]], device='cuda:0',
       grad_fn=<AddBackward0>)

In [ ]:
import torch.nn.functional as F
import math
from dataclasses import dataclass

B, T, C = 1, 4, 2

X = torch.tensor([
    [[0.1, 0.1],    # A
     [1.0, 0.2],    # crane
     [0.1, 0.9],    # ate
     [0.8, 0.0]]    # fish
]).float()

In [ ]:
X.shape

torch.Size([1, 4, 2])

In [ ]:
q_proj = nn.Linear(C, C, bias=False)
k_proj = nn.Linear(C, C, bias=False)
v_proj = nn.Linear(C, C, bias=False)

In [ ]:
torch.manual_seed(42)
q_proj.weight.data = torch.randn(C, C)
k_proj.weight.data = torch.randn(C, C)
v_proj.weight.data = torch.randn(C, C)

In [ ]:
q = q_proj(X)
k = k_proj(X)
v = v_proj(X)

In [ ]:
q.shape

torch.Size([1, 4, 2])

In [ ]:
q

tensor([[[ 0.2355,  0.0677],
         [ 0.6263, -0.0022],
         [ 1.9646,  0.7469],
         [ 0.1552, -0.1376]]], device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [ ]:
d = q.transpose(0, 2)
d

tensor([[[ 0.2355],
         [ 0.6263],
         [ 1.9646],
         [ 0.1552]],

        [[ 0.0677],
         [-0.0022],
         [ 0.7469],
         [-0.1376]]], device='cuda:0', grad_fn=<TransposeBackward0>)

In [ ]:
d.shape

torch.Size([2, 4, 1])

In [ ]:
scores = q @ k.transpose(-2, -1)
scores

tensor([[[ 9.9388e-04, -1.0651e-02,  2.1583e-02, -1.2638e-02],
         [ 1.9278e-03,  7.4853e-02, -5.3647e-02,  7.0997e-02],
         [ 9.0048e-03, -1.9201e-01,  2.9106e-01, -2.1002e-01],
         [-5.9973e-05,  9.6154e-02, -9.6814e-02,  9.6274e-02]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)

In [ ]:
d_k = k.size(-1)
scaled_scores = scores / math.sqrt(d_k)

In [ ]:
scaled_scores.size()

torch.Size([1, 4, 4])

### **We will debug whole LLM code**

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
torch.cuda.is_available()

True

In [ ]:
torch.set_default_device("cuda")

In [ ]:
tokens = torch.randint(0, 1000, (2, 4))

In [ ]:
tokens

tensor([[981, 197, 377, 811],
        [641, 896, 709, 674]], device='cuda:0')

In [ ]:
B, T = tokens.size()

In [ ]:
print(tokens.device)

cuda:0


In [ ]:
pos = torch.arange(0, T, dtype=torch.long).unsqueeze(0)

In [ ]:
pos

tensor([[0, 1, 2, 3]], device='cuda:0')

In [7]:
# gpt2_min.py
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import pdb

@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head, self.n_embd = config.n_head, config.n_embd
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.resid_drop = nn.Dropout(config.dropout)
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size)).view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        # pdb.set_trace()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_dim))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # self.lm_head.weight = self.wte.weight
    def forward(self, idx, targets=None):
        B, T = idx.size()
        # pdb.set_trace()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.wte(idx) + self.wpe(pos)
        x = self.drop(x)
        for block in self.h:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        # pdb.set_trace()
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        # pdb.set_trace()
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            # pdb.set_trace()
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                thresh = v[:, -1].unsqueeze(-1)
                logits = torch.where(logits < thresh, torch.full_like(logits, -float("inf")), logits)

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            # pdb.set_trace()
            idx = torch.cat((idx, next_token), dim=1)
        return idx

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Transformers are amazing"

tokens = tokenizer.encode(text)

print(tokens)

print(tokenizer.decode(tokens))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[41762, 364, 389, 4998]
Transformers are amazing


In [ ]:
AutoTokenizer

transformers.models.auto.tokenization_auto.AutoTokenizer

In [12]:
config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=1024,
    n_layer=8,
    n_head=8,
    n_embd=512,
    dropout=0.1
)

In [13]:
device = "cuda"

In [14]:
model = GPT(config).to(device)

In [15]:
model.parameters

<bound method Module.parameters of GPT(
  (wte): Embedding(50257, 512)
  (wpe): Embedding(1024, 512)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-7): 8 x Block(
      (ln_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=512, out_features=1536, bias=True)
        (c_proj): Linear(in_features=512, out_features=512, bias=True)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc): Linear(in_features=512, out_features=2048, bias=True)
        (proj): Linear(in_features=2048, out_features=512, bias=True)
        (drop): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=50257, bias=False)
)>

In [16]:
print(torch.numel)

<built-in method numel of type object at 0x7c73ebee5180>


In [17]:
num_params = sum(p.numel() for p in model.parameters())

print(f"{num_params/1e6:.2f}M parameters")

77.21M parameters


## **Generate random text to check model**

In [25]:
text = "The cat sat on the mat"
tokens = tokenizer.encode(text)

x = torch.tensor([tokens], device=device)

print(x.shape)

torch.Size([1, 6])


In [26]:
x

tensor([[ 464, 3797, 3332,  319,  262, 2603]], device='cuda:0')

In [30]:
text = "The cat sat on the mat"
tokens = tokenizer.encode(text)

x = torch.tensor([tokens], device=device)
out = model.generate(x, max_new_tokens=50)

print(tokenizer.decode(out[0].tolist()))

The cat sat on the matRyan Bachelor Hon supplementaryorld vault services Owners provincesucingworkshop allege Few phenotype bulls feeble similaritiesongevity� Nich lobbyingikingYetrg vo cited blasted mounts customization============ punkittal Off Zi Phar WolvesorientedBs KingsGROUP519usatophenOnt combo pioneersmanagement Were Osh Kinder


### **Training of Model**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

In [ ]:
tokenizer.pad_token

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def get_batch(data, block_size, batch_size):
  ix = torch.randint(len(data) - block_size - 1, (batch_size,))

  x = torch.stack([
      data[i:i+block_size] for i in ix
  ])

  y = torch.stack([data[i+1:i+block_size+1] for i in ix])

  return x.to(device), y.to(device)

In [ ]:
def tokenize(batch):
    # pdb.set_trace()
    result = tokenizer(batch["text"])

    result["input_ids"] = [
        ids + [tokenizer.eos_token_id]
        for ids in result["input_ids"]
    ]

    return result

In [ ]:
small_ds = dataset["train"]

In [ ]:
len(dataset["train"])

2119719

In [ ]:
print("Before tokenize:", dataset["train"][0])
print("------------------------------------------------")
print("after tokenize:", tokenize(dataset["train"][:1]))

Before tokenize: {'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}
------------------------------------------------
after tokenize: {'input_ids': [[3198, 1110, 11, 257, 1310, 2576, 3706, 20037, 1043, 257, 17598, 287, 607, 2119, 13, 1375, 2993, 340, 373, 2408, 284, 711, 351, 340, 780, 340, 373, 7786, 13, 20037, 2227, 284, 2648, 26

In [ ]:
small_ds.map

<bound method Dataset.map of Dataset({
    features: ['text'],
    num_rows: 2119719
})>

In [ ]:
tokenized = small_ds.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
all_ids = []

for ids in tokenized["input_ids"]:
    all_ids.extend(ids)

data = torch.tensor(all_ids, dtype=torch.long)

print(data.shape)

In [ ]:
import time

start = time.time()

xb, yb = get_batch(data, 128, 16)

logits, loss = model(xb, yb)

loss.backward()

print(time.time() - start)

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [ ]:
for step in range(50000):

    xb, yb = get_batch(
        data,
        block_size=64,
        batch_size=8
    )

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(f"step {step} loss {loss.item():.4f}")

    if step % 500 == 0:

        model.eval()

        prompt = "Once upon a time"

        tokens = tokenizer.encode(prompt)

        x = torch.tensor([tokens], device=device)

        out = model.generate(
            x,
            max_new_tokens=80,
            temperature=0.8
        )

        print(tokenizer.decode(out[0].tolist()))

        model.train()

        # save model in drive
        checkpoint = {
            "step": step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        }

        torch.save(
            checkpoint,
            "/content/latest8.pt"
        )

      # save permanent checkpoint every 5000 steps
    if step % 5000 == 0:
        torch.save(
            checkpoint,
            f"/content/drive/MyDrive/checkpoint8_{step}.pt"
        )

In [ ]:
prompt = "how are "

tokens = tokenizer.encode(prompt)

x = torch.tensor([tokens], device=device)

out = model.generate(
    x,
    max_new_tokens=40,
    temperature=0.8
)

print(tokenizer.decode(out[0].tolist()))

In [ ]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca")

In [ ]:
print(dataset["train"][0])

In [ ]:
def format_example(example):

    text = f"""User: {example['instruction']}

Assistant: {example['output']}"""

    return {"text": text}

In [ ]:
formatted = dataset["train"].map(
    format_example
)

In [ ]:
def tokenize(batch):

    result = tokenizer(
        batch["text"]
    )

    result["input_ids"] = [
        ids + [tokenizer.eos_token_id]
        for ids in result["input_ids"]
    ]

    return result

In [ ]:
tokenized = formatted.map(
    tokenize,
    batched=True,
    remove_columns=formatted.column_names
)

In [ ]:
tokenized[1]

In [ ]:
all_ids = []

for ids in tokenized["input_ids"]:
    all_ids.extend(ids)

In [ ]:
data = torch.tensor(
    all_ids,
    dtype=torch.long
)

print(data.shape)

In [ ]:
total_tokens = sum(
    len(ids)
    for ids in tokenized["input_ids"]
)

print(f"{total_tokens:,}")

In [ ]:
for step in range(6000):

    xb, yb = get_batch(
        data,
        block_size=64,
        batch_size=8
    )

    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(f"step {step} loss {loss.item():.4f}")

    if step % 500 == 0:

        model.eval()

        prompt = """
        User:What is Python?

        Assistant:
        """

        tokens = tokenizer.encode(prompt)

        x = torch.tensor([tokens], device=device)

        out = model.generate(
            x,
            max_new_tokens=80,
            temperature=0.8
        )

        print(tokenizer.decode(out[0].tolist()))

        model.train()

        # save model in drive
        checkpoint = {
            "step": step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        }

        torch.save(
            checkpoint,
            "/content/latest.pt"
        )

      # save permanent checkpoint every 5000 steps
    if step % 5000 == 0:
        torch.save(
            checkpoint,
            f"/content/drive/MyDrive/conversation_{step}.pt"
        )

In [ ]:
dataset["train"][5]

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------------------------------
# Dataset
# --------------------------------------------------

dataset = load_dataset(
    "tatsu-lab/alpaca",
    split="train",
    streaming=True
)

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# --------------------------------------------------
# Format examples
# --------------------------------------------------

def format_example(example):

    text = f"""User: {example['instruction']}

Assistant: {example['output']}"""

    return text

# --------------------------------------------------
# Streaming Dataset
# --------------------------------------------------

class GPTStreamDataset(IterableDataset):

    def __init__(self, dataset, tokenizer, block_size):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.block_size = block_size

    def __iter__(self):

        buffer = []

        for example in self.dataset:

            text = format_example(example)

            ids = self.tokenizer.encode(text)

            ids.append(
                self.tokenizer.eos_token_id
            )

            buffer.extend(ids)

            while len(buffer) >= self.block_size + 1:

                x = torch.tensor(
                    buffer[:self.block_size],
                    dtype=torch.long
                )

                y = torch.tensor(
                    buffer[1:self.block_size + 1],
                    dtype=torch.long
                )

                yield x, y

                buffer = buffer[self.block_size:]

# --------------------------------------------------
# Config
# --------------------------------------------------

config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=128,
    n_layer=8,
    n_head=8,
    n_embd=512,
    dropout=0.1
)

model = GPT2(config).to(device)

# --------------------------------------------------
# DataLoader
# --------------------------------------------------

stream_ds = GPTStreamDataset(
    dataset,
    tokenizer,
    block_size=config.block_size
)

loader = DataLoader(
    stream_ds,
    batch_size=4
)

# --------------------------------------------------
# Optimizer
# --------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

# --------------------------------------------------
# Training
# --------------------------------------------------

model.train()

for step, (xb, yb) in enumerate(loader):

    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(
        xb,
        yb
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    optimizer.step()

    if step % 100 == 0:

        print(
            f"step={step} "
            f"loss={loss.item():.4f}"
        )

    if step == 10000:
        break

# --------------------------------------------------
# Save
# --------------------------------------------------

torch.save(
    model.state_dict(),
    "alpaca_gpt.pt"
)

step=0 loss=11.0494
step=100 loss=6.0155
step=200 loss=7.0330
step=300 loss=6.0750
step=400 loss=5.6852
step=500 loss=9.7688
step=600 loss=5.5677
step=700 loss=5.4555
step=800 loss=5.1452
step=900 loss=5.9735
step=1000 loss=5.8118
step=1100 loss=5.5389
step=1200 loss=5.4127
step=1300 loss=4.6185
step=1400 loss=6.4405
step=1500 loss=5.0375
step=1600 loss=5.2281
step=1700 loss=5.0145
step=1800 loss=5.7677
step=1900 loss=4.9587
step=2000 loss=5.2121
step=2100 loss=4.8230
step=2200 loss=4.9804
step=2300 loss=5.0993
step=2400 loss=4.4013
step=2500 loss=4.5740
step=2600 loss=4.6233
step=2700 loss=3.8194
step=2800 loss=5.7264
step=2900 loss=4.5403
step=3000 loss=4.7097
step=3100 loss=4.7413
step=3200 loss=5.1953
step=3300 loss=5.2485
step=3400 loss=5.0479
step=3500 loss=5.4154
step=3600 loss=4.7881
step=3700 loss=4.4157


Token indices sequence length is longer than the specified maximum sequence length for this model (1486 > 1024). Running this sequence through the model will result in indexing errors


step=3800 loss=4.7376
step=3900 loss=4.5898
step=4000 loss=5.1663
step=4100 loss=3.4418
step=4200 loss=4.5491
step=4300 loss=4.2701
step=4400 loss=4.8830
step=4500 loss=4.1107
step=4600 loss=4.5118
step=4700 loss=4.1627
step=4800 loss=4.5705
step=4900 loss=3.3654
step=5000 loss=4.3337
step=5100 loss=4.9992
step=5200 loss=4.7004
step=5300 loss=3.9855
step=5400 loss=4.9116
step=5500 loss=4.1225
step=5600 loss=4.8869
step=5700 loss=4.7827
step=5800 loss=4.1968
step=5900 loss=4.2306
step=6000 loss=4.2130
step=6100 loss=4.8788
step=6200 loss=3.7390
step=6300 loss=4.6307
step=6400 loss=3.7457
step=6500 loss=3.8013
step=6600 loss=3.5482
step=6700 loss=3.9517
step=6800 loss=4.2537
step=6900 loss=4.1468
step=7000 loss=4.5283
step=7100 loss=4.1779
step=7200 loss=4.0027
step=7300 loss=4.9429
step=7400 loss=4.5339
step=7500 loss=4.4160
step=7600 loss=3.5367
step=7700 loss=4.4533


In [ ]:
torch.save(
        {
            "step": step,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        },
        "/content/drive/MyDrive/Trained-Model/gpt_8.pt"
    )

In [ ]:
model.eval()

@torch.no_grad()
def generate_text(prompt, max_new_tokens=100):

    ids = tokenizer.encode(prompt)

    x = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    out = model.generate(
        x,
        max_new_tokens=max_new_tokens,
        temperature=0.8,
        top_k=50
    )

    return tokenizer.decode(
        out[0].tolist()
    )

In [ ]:
print(
    generate_text(
        "Once upon a time"
    )
)

Once upon a time and deep learning.

Assistant: One example of a deep learning algorithm is that the input data is the end of data.<|endoftext|>User: Name five elements that are commonly used in machine learning.

Assistant: 1. Machine Learning techniques such as machine learning, computer vision, images, machine learning, and machine learning algorithms. 
2. Machine learning algorithms that are commonly used to create automated data points and data.
3. Decision-Learnans models to make predictions more data to


In [ ]:
print(
    generate_text(
        "User: What is machine learning?\n\nAssistant:"
    )
)

User: What is machine learning?

Assistant: Vector

Assistant: Machine Learning is a tool that works to learn and track a computer program. Machine learning algorithms can be used to describe a variety of images and generate more natural language processing. Machine learning algorithms can often detect outliers in text processing models, while Python is a programming language model that can be used for text data, and can be trained to recognize patterns and make predictions about their models. Vector Machine learning algorithms can be used to classify their images of text and recognize patterns in their operations


In [ ]:
print(
    generate_text(
        "User: What is Python?\n\nAssistant:"
    )
)

User: What is Python?

Assistant: Python is the process of using machine learning techniques. Python is a machine learning algorithm that accurately identify patterns in different fields or patterns. Machine Learning is used to identify patterns and identify patterns in a machine learning model to find the text.<|endoftext|>User: Create a story about a young man who was the first person who had a little witch.

Assistant: Once upon a time, there was a young bear named Sarah who lived in the park.<|endoftext|>User: Explain the steps involved in the blank.


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
import torch
from torch.utils.data import IterableDataset

class TinyStoriesStreamDataset(IterableDataset):

    def __init__(
        self,
        dataset,
        tokenizer,
        block_size=128
    ):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.block_size = block_size

    def __iter__(self):

        buffer = []

        for example in self.dataset:

            text = example["text"]

            ids = self.tokenizer.encode(text)

            ids.append(
                tokenizer.eos_token_id
            )

            buffer.extend(ids)

            while len(buffer) >= self.block_size + 1:

                x = torch.tensor(
                    buffer[:self.block_size],
                    dtype=torch.long
                )

                y = torch.tensor(
                    buffer[1:self.block_size + 1],
                    dtype=torch.long
                )

                yield x, y

                buffer = buffer[
                    self.block_size:
                ]

In [ ]:
from torch.utils.data import DataLoader

stream_ds = TinyStoriesStreamDataset(
    dataset,
    tokenizer,
    block_size=128
)

loader = DataLoader(
    stream_ds,
    batch_size=4
)

In [ ]:
num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"{num_params/1e6:.2f}M parameters"
)

76.75M parameters


In [ ]:
model.train()

for step, (xb, yb) in enumerate(loader):

    xb = xb.to(device)
    yb = yb.to(device)

    logits, loss = model(
        xb,
        yb
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    optimizer.step()

    if step % 100 == 0:

        print(
            f"step={step} "
            f"loss={loss.item():.4f}"
        )

    if step % 1000 == 0 and step > 0:

        torch.save(
            {
                "step": step,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
            },
            "/content/drive/MyDrive/Trained-Model/tinystories.pt"
        )

    if step % 10000 == 0 and step > 0:

        torch.save(
            {
                "step": step,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
            },
            "/content/drive/MyDrive/Trained-Model/tinystoriesL.pt"
        )

    if step >= 50000:
        break

step=0 loss=5.3595
step=100 loss=3.7218
step=200 loss=3.8851
step=300 loss=3.3692
step=400 loss=3.2959
step=500 loss=3.9561
step=600 loss=3.5267
step=700 loss=3.8102
step=800 loss=3.1213
step=900 loss=3.5331


Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


step=1000 loss=2.8683
step=1100 loss=3.6608
step=1200 loss=3.3840
step=1300 loss=3.3749
step=1400 loss=3.1925
step=1500 loss=2.9757
step=1600 loss=3.2660
step=1700 loss=2.7953
step=1800 loss=3.6806
step=1900 loss=3.3126
step=2000 loss=2.7580
step=2100 loss=2.9498
step=2200 loss=3.5153
step=2300 loss=3.0372
step=2400 loss=2.6542
step=2500 loss=3.2856
step=2600 loss=3.1780
step=2700 loss=2.4674
step=2800 loss=2.5739
step=2900 loss=2.9278
step=3000 loss=2.2357
step=3100 loss=2.7554
step=3200 loss=2.7843
step=3300 loss=2.9418
step=3400 loss=2.6527
step=3500 loss=3.4097
step=3600 loss=2.7172
step=3700 loss=3.0803
step=3800 loss=2.9254
step=3900 loss=2.3931
step=4000 loss=2.8213
step=4100 loss=2.7629
step=4200 loss=3.0899
step=4300 loss=2.6457
step=4400 loss=1.9769
step=4500 loss=3.1016
step=4600 loss=2.5211
step=4700 loss=2.5983
step=4800 loss=2.2522
step=4900 loss=2.6224
step=5000 loss=3.5683
step=5100 loss=2.5437
step=5200 loss=3.0271
step=5300 loss=3.7566
step=5400 loss=2.3409
step=5500 

In [ ]:
torch.save(
            {
                "step": step,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
            },
            "/content/drive/MyDrive/Trained-Model/tinystoriesFinal.pt"
        )

In [ ]:
print(
    generate_text(
        """
        User: Tell me a story about a dragon.

Assistant: ..."""
    )
)


        User: Tell me a story about a dragon.

Assistant: ..., I don't think about how much your story is about a brave knight! You must be brave, and you will find out what you are. And guess what?"

Sara thought that sounded like a real dragon. She understood that her book had her own secret, and that she was glad she had answered it with her.<|endoftext|>Once upon a time, there was a little mouse named Timmy. Timmy was a big bear who lived in the forest. One day, Timmy


In [ ]:
del dataset
del stream_ds
del loader

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "stingning/ultrachat",
    split="train",
    streaming=True
)

README.md:   0%|          | 0.00/3.12k [00:00<?, ?B/s]

In [ ]:
sample = next(iter(dataset))
print(sample)

{'id': '0', 'data': ['How can cross training benefit groups like runners, swimmers, or weightlifters?', 'Cross training can benefit groups like runners, swimmers, or weightlifters in the following ways:\n\n1. Reduces the risk of injury: Cross training involves different types of exercises that work different muscle groups. This reduces the risk of overuse injuries that may result from repetitive use of the same muscles.\n\n2. Improves overall fitness: Cross training helps improve overall fitness levels by maintaining a balance of strength, endurance, flexibility, and cardiovascular fitness.\n\n3. Breaks monotony: Cross training adds variety to your fitness routine by introducing new exercises, which can help you stay motivated and avoid boredom that often comes with doing the same exercises repeatedly.\n\n4. Increases strength: Cross training helps in building strength by incorporating exercises that target different muscle groups. This helps you build strength in areas that may be und

In [ ]:
sample = next(iter(dataset))

print(type(sample))
print(sample)

for k, v in sample.items():
    print("\nKEY:", k)
    print(type(v))
    print(v)

<class 'dict'>
{'id': '0', 'data': ['How can cross training benefit groups like runners, swimmers, or weightlifters?', 'Cross training can benefit groups like runners, swimmers, or weightlifters in the following ways:\n\n1. Reduces the risk of injury: Cross training involves different types of exercises that work different muscle groups. This reduces the risk of overuse injuries that may result from repetitive use of the same muscles.\n\n2. Improves overall fitness: Cross training helps improve overall fitness levels by maintaining a balance of strength, endurance, flexibility, and cardiovascular fitness.\n\n3. Breaks monotony: Cross training adds variety to your fitness routine by introducing new exercises, which can help you stay motivated and avoid boredom that often comes with doing the same exercises repeatedly.\n\n4. Increases strength: Cross training helps in building strength by incorporating exercises that target different muscle groups. This helps you build strength in areas 

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "gpt2"
)

In [ ]:
import torch
from torch.utils.data import IterableDataset

class UltraChatStreamDataset(
    IterableDataset
):

    def __init__(
        self,
        dataset,
        tokenizer,
        block_size=128
    ):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.block_size = block_size

    def __iter__(self):

        buffer = []

        for example in self.dataset:

            messages = example["data"]

            text = ""

            for msg in messages:

                role = msg["role"]
                content = msg["content"]

                if role == "user":

                    text += (
                        f"User: {content}\n\n"
                    )

                elif role == "assistant":

                    text += (
                        f"Assistant: {content}\n\n"
                    )

            ids = self.tokenizer.encode(
                text
            )

            ids.append(
                self.tokenizer.eos_token_id
            )

            buffer.extend(ids)

            while (
                len(buffer)
                >= self.block_size + 1
            ):

                x = torch.tensor(
                    buffer[:self.block_size],
                    dtype=torch.long
                )

                y = torch.tensor(
                    buffer[
                        1:self.block_size + 1
                    ],
                    dtype=torch.long
                )

                yield x, y

                buffer = buffer[
                    self.block_size:
                ]

In [1]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from huggingface_hub import create_repo

create_repo(
    repo_id="Dipak654/gpt-8-layer",
    private=False
)

RepoUrl('https://huggingface.co/Dipak654/gpt-8-layer', endpoint='https://huggingface.co', repo_type='model', repo_id='Dipak654/gpt-8-layer')

In [6]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="/content/drive/MyDrive/Trained-Model/tinystoriesFinal.pt",
    path_in_repo="tinystoriesFinal.pt",
    repo_id="Dipak654/gpt-8-layer"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...Model/tinystoriesFinal.pt:   0%|          |  557kB /  922MB            

CommitInfo(commit_url='https://huggingface.co/Dipak654/gpt-8-layer/commit/94b1ab713e646b3848b6da3c04baba7452fff646', commit_message='Upload tinystoriesFinal.pt with huggingface_hub', commit_description='', oid='94b1ab713e646b3848b6da3c04baba7452fff646', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Dipak654/gpt-8-layer', endpoint='https://huggingface.co', repo_type='model', repo_id='Dipak654/gpt-8-layer'), pr_revision=None, pr_num=None)